# Mixed-CUTS on Kaggle (T4)

Runs the repo's Kaggle scripts (decision 014): a smoke test, one training arm, or evaluations.

**Notebook settings** (right panel): Accelerator **GPU T4 x2** (not P100: CUDA 13 / vLLM do not support it),
**Internet on** (needs a phone-verified account), and these **Secrets** (Add-ons -> Secrets):
- `WANDB_API_KEY`: your W&B key (the runs log to `anlp-mixed-cuts/mixed-cuts`)
- `HF_TOKEN`: a Hugging Face token with **write** access (gated GPQA download + the run carry-over repo)

**How a run spans sessions**: a session lasts at most 12 hours and loses its disk. `kaggle/train.sh` pulls the
run's newest checkpoint from the private Hub repo `HUB_REPO`, trains, pushes every checkpoint, and stops
cleanly before the limit. To continue, run this notebook again. Use **Save Version -> Save & Run All
(Commit)** so it runs in the background without the browser open. Every session costs weekly GPU quota
(~30 h), whether it uses one T4 or two.

In [ ]:
# ---- parameters ------------------------------------------------------------------------------------
MODE = "smoke"             # "smoke" (~20 min check) | "train" (one arm) | "eval"
ARM = "math_mixed_cuts"    # train: math_grpo or math_mixed_cuts -> run name <arm>-t4-s<SEED>
SEED = 1
EVAL_TARGETS = ["base"]    # eval: up to two at once (one per T4): "base", "math_grpo-t4-s1", "math_grpo-h100-s1", ...
HUB_REPO = "CHANGE-ME/mixed-cuts-runs"   # private HF model repo that carries runs between sessions
WANDB_ENTITY = "anlp-mixed-cuts"
BRANCH = "main"

In [ ]:
# ---- session start, repo, secrets -> .env (values are never printed) ---------------------------------
import os, subprocess, time
os.environ["MC_SESSION_START_EPOCH"] = str(int(time.time()) - 120)   # the session started a little earlier
REPO_DIR = "/tmp/mixed-cuts"   # outside /kaggle/working on purpose: the venv would blow its 20 GB output limit
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/dumbkin31/exploration-in-grpo.git", REPO_DIR], check=True)
from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()
def _secret(name):
    try:
        return _secrets.get_secret(name)
    except Exception:
        return ""
env = {"WANDB_API_KEY": _secret("WANDB_API_KEY"), "HF_TOKEN": _secret("HF_TOKEN"),
       "WANDB_ENTITY": WANDB_ENTITY, "MC_HUB_REPO": HUB_REPO if "CHANGE-ME" not in HUB_REPO else ""}
with open(f"{REPO_DIR}/.env", "w") as f:
    f.writelines(f"{k}={v}\n" for k, v in env.items() if v)
print({k: ("set" if v else "MISSING") for k, v in env.items()})

In [ ]:
# ---- helper: run a script, stream its output ---------------------------------------------------------
def sh(cmd):
    p = subprocess.Popen(["bash", "-c", cmd], cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"{cmd!r} exited with {p.returncode}")

sh("bash kaggle/setup.sh")   # ~10-15 min, every session

In [ ]:
# ---- the work ------------------------------------------------------------------------------------------
if MODE == "smoke":
    sh("bash kaggle/smoke.sh 0")
elif MODE == "train":
    sh(f"bash kaggle/train.sh {ARM} {SEED} 0")      # one arm per session: host RAM fits one training run
elif MODE == "eval":
    jobs = [f"bash kaggle/eval.sh {t} {gpu} > /kaggle/working/eval-{gpu}.log 2>&1" for gpu, t in enumerate(EVAL_TARGETS[:2])]
    sh(" & ".join(jobs) + " & wait; tail -n 25 /kaggle/working/eval-*.log")
else:
    raise ValueError(MODE)

In [ ]:
# ---- what is on the Hub now, and what this session saved ----------------------------------------------
sh("MC_SITE=kaggle; source configs/ada.env.sh; .venv/bin/python scripts/hub_sync.py status || true; ls -R /kaggle/working/outputs | head -50")